# **The mask as a trainable parameter: Meaningful perturbation, Extremal perturbations, Real-time saliency and L2X**

Practice for the module [“Explanation via Perturbation”](https://ai-interpretability.school), lesson “Learnable baselines”.

In the practice `PRACTICE_perturbation_rise` the masks were enumerated: block by block in ablation, window by window in occlusion, one random mask after another in RISE. In the lesson “Learnable baselines” the mask is not enumerated but searched for: the mask is a tensor, the objective is two lines, and the rest is an ordinary optimizer. Here we check this on the same model and photo as in `PRACTICE_perturbation_rise` (ResNet-50 and the photo of a pig), and then move the optimization from explanation time to training time on MNIST.

1. Meaningful perturbation by hand: the deletion game, a mask on `torch.optim.Adam`, and what happens without regularization.
2. Extremal perturbations with the `torchray` library: masks of a given area of 5, 10 and 20%.
3. Comparison with occlusion from the lesson “Explaining DL models via perturbation: Practice” by deletion and insertion.
4. Real-time saliency on MNIST: a masking network that outputs the mask in one pass, and what one explanation costs.
5. L2X on MNIST: choosing $k$ blocks through the Concrete distribution and checking whose behavior the chosen subset describes.

Optimizing a mask on ResNet-50 takes hundreds of forward and backward passes per explanation. Switch Colab to a GPU runtime (“Runtime → Change runtime type → T4 GPU”): on a CPU, sections 1–3 take several times longer.

Piglet photo: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), Pexels License.

In [ ]:
!pip install -q torchray==1.0.0.2

In [ ]:
import time
import warnings
from io import BytesIO

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights
from torchray.attribution.extremal_perturbation import extremal_perturbation, simple_reward

warnings.filterwarnings('ignore', message='torch.meshgrid')   # a warning from inside torchray
torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad_(False)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
RAW = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/'

def load(name):
    return Image.open(BytesIO(requests.get(RAW + name).content)).convert('RGB')

image = load('piglet.jpg').resize((224, 224))
x = transform(image).unsqueeze(0).to(DEVICE)
blur = transform(image.filter(ImageFilter.GaussianBlur(12))).unsqueeze(0).to(DEVICE)   # the baseline of PRACTICE_perturbation_rise
categories = [s.strip() for s in requests.get(RAW + 'imagenet_classes.txt').text.splitlines()]

def predict(batch):
    with torch.no_grad():
        return torch.cat([model(batch[s:s + 32]).softmax(1)[:, target] for s in range(0, len(batch), 32)])

with torch.no_grad():
    target = model(x).argmax().item()
base_score = predict(x).item()
print(f'device: {DEVICE}')
print(f'class: {target} — {categories[target]}, confidence {base_score:.3f}')

## 1. Meaningful perturbation: the deletion game

The statement from the lesson “Learnable baselines” ([Fong and Vedaldi, 2017](https://arxiv.org/abs/1704.03296)):

$$m^{\ast}=\arg\min_{m\in[0,1]^{\Lambda}}\; f_c(\Phi(x,m))+\lambda\,\mathrm{mean}(1-m),\qquad \Phi(x,m)=x\odot m+b\odot(1-m).$$

Here $m$ is the mask, a one at a position means “keep”, a zero means “close”; $b$ is the blurred photo, the same baseline as in `PRACTICE_perturbation_rise`; $f_c$ is the confidence of ResNet-50 in the class hog; $\lambda$ is the price of closing a unit of area. The area penalty is written through the mean rather than the sum, so that $\lambda$ does not depend on the size of the picture.

How this differs from the paper. In Fong and Vedaldi the “blur” perturbation is a blur with a kernel whose width at every point depends on the mask (the largest standard deviation is $\sigma_0=10$), the penalties are written as sums with $\lambda_1=10^{-4}$ and $\lambda_2=10^{-2}$, the mask is blurred again after upsampling with a kernel $\sigma_m=5$, and the initial mask is the smallest circle that suppresses the confidence by 99%. Here the perturbation is a mix with one blurred photo `blur`: the same baseline as in RISE, occlusion and the metrics of section 3, otherwise they cannot be compared. The other differences do not change the method: writing the penalties as means only changes the scale of $\lambda$, while a mask of ones at the start and jitter of the perturbed input instead of jitter of the mask are details of the optimization.

The function `deletion_game` is the whole method. The mask starts as ones, the weights of the model are frozen, an Adam step changes only the mask. The three defenses against adversarial patterns from the step “Adversariality and masks” are switched on by arguments:

- `coarse` — a mask on a `coarse` $\times$ `coarse` grid, stretched to $224\times224$ by bilinear interpolation; `None` — a full-resolution mask;
- `tv` — the weight of the total variation penalty (Fong and Vedaldi raise the jump to the power $\beta=3$);
- `jitter` — a random shift of the perturbed input by up to 4 pixels at every step.

In [ ]:
def upsample(m, coarse):
    return m if coarse is None else F.interpolate(m, size=(224, 224), mode='bilinear', align_corners=False)

def deletion_game(lam, coarse=28, tv=2.0, jitter=True, steps=300, seed=0):
    """argmin_m f_c(Phi(x, m)) + lam * mean(1 - m) + tv * TV(m), Phi is a blend with the photo blur."""
    torch.manual_seed(seed)
    side = 224 if coarse is None else coarse
    m = torch.ones(1, 1, side, side, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([m], lr=0.1)
    for _ in range(steps):
        up = upsample(m, coarse)
        mixed = x * up + blur * (1 - up)
        if jitter:
            dy, dx = [int(v) for v in torch.randint(-4, 5, (2,))]
            mixed = torch.roll(mixed, shifts=(dy, dx), dims=(2, 3))
        loss = model(mixed).softmax(1)[0, target] + lam * (1 - up).mean()
        if tv:
            loss = loss + tv * (((up[..., 1:, :] - up[..., :-1, :]).abs() ** 3).mean()
                                + ((up[..., :, 1:] - up[..., :, :-1]).abs() ** 3).mean())
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            m.clamp_(0, 1)
    with torch.no_grad():
        up = upsample(m, coarse)
        score = predict(x * up + blur * (1 - up)).item()
    return up[0, 0].detach(), score, (1 - up).mean().item()

t0 = time.time()
mask_raw, f_raw, closed_raw = deletion_game(0.5, coarse=None, tv=0.0, jitter=False)
mask_reg, f_reg, closed_reg = deletion_game(0.5)
print(f'two runs of 300 steps: {time.time() - t0:.0f} s on {DEVICE}')
print(f'{"mask":44}{"area closed":>18}{"confidence after blur":>26}')
print(f'{"full resolution, no penalties":44}{closed_raw:>18.1%}{f_raw:>26.3f}')
print(f'{"28x28 grid, total variation, jitter":44}{closed_reg:>18.1%}{f_reg:>26.3f}')

In [ ]:
def show_mask(ax, closed, title):
    ax.imshow(image)
    ax.imshow(closed.cpu().numpy(), cmap='magma', alpha=0.6, vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image); axes[0].set_title(f'photo, confidence {base_score:.2f}', fontsize=10); axes[0].axis('off')
show_mask(axes[1], 1 - mask_raw, f'no regularization: closed {closed_raw:.1%}, f = {f_raw:.2f}')
show_mask(axes[2], 1 - mask_reg, f'with regularization: closed {closed_reg:.1%}, f = {f_reg:.2f}')
plt.tight_layout()
plt.show()

Bright in the pictures is what is closed, $1-m$. Both masks solve the optimization problem: the confidence in the class hog falls almost to zero. The mask without regularization does it with a scatter of single pixels on a tiny share of the area — an adversarial pattern, not an explanation: such pixels carry nothing a person would call a pig. The mask on a coarse grid with the total variation penalty and jitter closes connected patches on the animal's head. Regularization here is part of the definition of the method: without it the method answers a question about the fragility of the network, not about where the class is in the photo.

**Task 1.** Build the family of masks over $\lambda$, as in the step “Example” of the lesson “Learnable baselines”: run `deletion_game` with regularization at $\lambda\in\{20,\,8,\,1,\,0.1\}$ and for each value print the share of the area closed and the confidence after blurring. At which $\lambda$ is the mask empty? Are the masks nested in one another, as in the toy example? Check the nesting by the share of pixels with $1-m>0.5$ that are closed at a larger $\lambda$ but open at a smaller one.

In [ ]:
# Your code here

## 2. Extremal perturbations: the area is fixed in advance

In Extremal perturbations ([Fong, Patrick, and Vedaldi, 2019](https://arxiv.org/abs/1910.08485)) the game is the opposite one, the preservation game: find the mask of a given area $a$ under which the class is preserved best, with everything outside the mask blurred. Smoothness is built into the family of masks (a mask is built from a coarse grid with step 7 and smoothing), and the area is held by a penalty on the sorted values of the mask. All of this is inside `torchray.attribution.extremal_perturbation`. The arguments:

- `areas` — a list of area shares, one mask per share; we take 5, 10 and 20%;
- `reward_func=simple_reward` — the objective of the paper: the class score before softmax; it is also the default of the library;
- `max_iter=800` — the number of optimization steps, the default of the library. The paper uses 1600 steps of SGD with momentum and doubles the weight of the area penalty at one third and two thirds of the way; `torchray` by default takes 800 steps and grows the penalty weight smoothly, by a factor of 1.0035 per step.

We compute the preserved confidence the same way as in section 1: the photo under the mask, the blurred photo `blur` outside it.

In [ ]:
AREAS = [0.05, 0.1, 0.2]
torch.manual_seed(0)
t0 = time.time()
ep_masks, _ = extremal_perturbation(model, x, target, areas=AREAS, reward_func=simple_reward, max_iter=800)
ep_masks = ep_masks[:, 0].detach()
print(f'three masks in {time.time() - t0:.0f} s on {DEVICE}')
print(f'{"target area":>18}{"mask mean":>16}{"share m > 0.5":>16}{"confidence under mask":>26}')
for a, m in zip(AREAS, ep_masks):
    kept = predict(x * m + blur * (1 - m)).item()
    print(f'{a:>18.0%}{m.mean().item():>16.3f}{(m > 0.5).float().mean().item():>16.3f}{kept:>26.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, a, m in zip(axes, AREAS, ep_masks):
    ax.imshow(np.asarray(image) / 255 * (0.2 + 0.8 * m.cpu().numpy()[..., None]))
    ax.set_title(f'area {a:.0%}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

Bright here is what the mask has kept. Instead of one map we get a family, and it is read from the table. The mean of the mask shows how closely the penalty held the target area, the share of pixels with $m>0.5$ shows how binary the mask is, and the last column shows whether this area is enough for the class. On this photo the confidence under the mask grows with the area: 0.115 at 5%, 0.254 at 10% and 0.359 at 20%, less than half of the original 0.926. The class hog comes back gradually, and the area at which it is fully preserved is not in this sweep. This is how Fong, Patrick, and Vedaldi propose to read the family.

**Task 2.** Compare the answers of the two games. Take the Extremal perturbations mask at area 10% (pixels with $m>0.5$) and the 10% of pixels most strongly closed by the regularized Meaningful perturbation mask from section 1 (`1 - mask_reg`). Compute the IoU of the two sets and the Pearson correlation of the maps `ep_masks[1]` and `1 - mask_reg`. Do the preservation game and the deletion game give the same answer?

In [ ]:
# Your code here

## 3. Comparison with occlusion by deletion and insertion

The metrics are the same as in section 5 of `PRACTICE_perturbation_rise`: deletion removes pixels in decreasing order of importance and watches the confidence fall (a lower area is better), insertion returns pixels into the blurred photo (higher is better). The importance map of Meaningful perturbation is the closedness $1-m$; that of Extremal perturbations is the sum of the three masks, as in the paper: a pixel kept already at 5% of the area is more important than a pixel that appears only at 20%. Occlusion is a window of 64 with stride 16, as in `PRACTICE_perturbation_rise`, and a random map checks the metrics.

In [ ]:
def occlusion(size, stride):
    cells = [(i, j) for i in range(0, 224 - size + 1, stride) for j in range(0, 224 - size + 1, stride)]
    batch = x.repeat(len(cells), 1, 1, 1)
    for k, (i, j) in enumerate(cells):
        batch[k, :, i:i + size, j:j + size] = blur[0, :, i:i + size, j:j + size]
    scores = predict(batch)
    total, count = torch.zeros(224, 224, device=DEVICE), torch.zeros(224, 224, device=DEVICE)
    for (i, j), s in zip(cells, scores):
        total[i:i + size, j:j + size] += base_score - s
        count[i:i + size, j:j + size] += 1
    return total / count

def curve_auc(heat, mode, steps=100):
    """deletion: move pixels of the photo into the blur; insertion: return the photo into the blur."""
    order = heat.flatten().argsort(descending=True)
    per = len(order) // steps
    cur = (x if mode == 'deletion' else blur).clone()
    src = blur[0] if mode == 'deletion' else x[0]
    curve = [predict(cur).item() / base_score]
    for t in range(steps):
        idx = order[t * per:(t + 1) * per]
        rows, cols = idx // 224, idx % 224
        cur[0, :, rows, cols] = src[:, rows, cols]
        curve.append(predict(cur).item() / base_score)
    return float(np.trapezoid(curve, dx=1 / steps))

torch.manual_seed(0)
heats = {'Meaningful perturbation, 1 - m': 1 - mask_reg,
         'Extremal perturbations, sum of masks': ep_masks.sum(0),
         'Occlusion, window 64, stride 16': occlusion(64, 16),
         'random map': torch.rand(224, 224, device=DEVICE)}
print(f'{"map":38}{"deletion (lower is better)":>28}{"insertion (higher is better)":>30}')
for name, heat in heats.items():
    print(f'{name:38}{curve_auc(heat, "deletion"):>28.4f}{curve_auc(heat, "insertion"):>30.4f}')

Both metrics use the blur `blur` as the baseline. By deletion occlusion leads (0.0443 against 0.0619 for Extremal perturbations and 0.0787 for Meaningful perturbation), while by insertion Extremal perturbations leads: 0.7342 against 0.6952 for Meaningful perturbation and 0.6469 for occlusion. Each optimization method searched for a mask for its own setting. Meaningful perturbation searched for a mask for a blend with the same `blur`, and by deletion — the question of the deletion game — its map is closer to occlusion than to the random one. Extremal perturbations searched for masks for the blur pyramid of `torchray` and for only three areas, while deletion and insertion go through all shares of the area from 0 to 100%. A map that is optimal for one setting does not have to be the best for another: before comparing methods by a number, check which question the metric answers and which perturbation it uses.

## 4. Real-time saliency: a mask in one pass

Every run in sections 1 and 2 is hundreds of forward and backward passes of ResNet-50 for one picture. Real-time saliency ([Dabkowski and Gal, 2017](https://arxiv.org/abs/1705.07857)) moves this optimization to training time: a separate masking network learns to output a mask from the image, and an explanation needs one forward pass of it. The objective from the lesson “Learnable baselines”:

$$L(M)=\lambda_1\,TV(M)+\lambda_2\,AV(M)-\log f_c\bigl(\Phi(X,M)\bigr)+\lambda_3\,f_c\bigl(\Phi(X,1-M)\bigr)^{\lambda_4}.$$

In the paper the masking network is trained on ImageNet on top of ResNet-50, which takes hours on a GPU. Here is the same objective on MNIST, where training takes minutes even on a CPU:

- the model being explained is a small convolutional network trained for one epoch; after that its weights are frozen;
- $\Phi(X,M)=X\odot M$: the background of MNIST is zero, and zero is the natural alternative $A$;
- the masking network outputs a mask on a $7\times7$ grid and stretches it to $28\times28$, the same coarse-grid defense as in section 1;
- the class $c$ is the one predicted by the model being explained, so the masking network does not need the class as input (in the paper it does, because a mask is requested for any class);
- $\lambda_1=1,\ \lambda_2=2,\ \lambda_3=5,\ \lambda_4=0.3$.

In [ ]:
mnist_train = datasets.MNIST('data', train=True, download=True)
mnist_test = datasets.MNIST('data', train=False, download=True)
Xtr = mnist_train.data.float().div(255).unsqueeze(1).to(DEVICE)
Xte = mnist_test.data.float().div(255).unsqueeze(1).to(DEVICE)
Ytr, Yte = mnist_train.targets.to(DEVICE), mnist_test.targets.to(DEVICE)

def batches(X, bs=128, shuffle=True):
    idx = torch.randperm(len(X), device=DEVICE) if shuffle else torch.arange(len(X), device=DEVICE)
    for s in range(0, len(X), bs):
        yield idx[s:s + bs]

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 32, 3, padding=1)
        self.c2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc = nn.Linear(64 * 7 * 7, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)
        x = F.max_pool2d(F.relu(self.c2(x)), 2)
        return self.out(F.relu(self.fc(x.flatten(1))))

torch.manual_seed(0)
clf = SmallCNN().to(DEVICE)
opt = torch.optim.Adam(clf.parameters(), 1e-3)
for j in batches(Xtr):
    loss = F.cross_entropy(clf(Xtr[j]), Ytr[j])
    opt.zero_grad()
    loss.backward()
    opt.step()
clf.eval()
for p in clf.parameters():
    p.requires_grad_(False)
with torch.no_grad():
    pred_te = torch.cat([clf(Xte[j]).argmax(1) for j in batches(Xte, 1000, False)])
print(f'accuracy of the explained model on the MNIST test set: {(pred_te == Yte).float().mean():.4f}')

In [ ]:
class Masker(nn.Module):
    """Image 28x28 -> mask 7x7 in (0, 1) -> stretched to 28x28."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 1, 1))

    def forward(self, x):
        return F.interpolate(torch.sigmoid(self.net(x)), size=(28, 28), mode='bilinear', align_corners=False)

def tv(m):
    return (((m[..., 1:, :] - m[..., :-1, :]) ** 2).mean((1, 2, 3))
            + ((m[..., :, 1:] - m[..., :, :-1]) ** 2).mean((1, 2, 3)))

def rts_loss(m, xb, c, l1=1.0, l2=2.0, l3=5.0, l4=0.3):
    """L(M) from the lesson; Phi(X, M) = X * M, the alternative is the zero background of MNIST."""
    keep = clf(xb * m).softmax(1).gather(1, c[:, None])[:, 0]
    drop = clf(xb * (1 - m)).softmax(1).gather(1, c[:, None])[:, 0]
    return l1 * tv(m) + l2 * m.mean((1, 2, 3)) - torch.log(keep + 1e-6) + l3 * (drop + 1e-6) ** l4

torch.manual_seed(0)
masker = Masker().to(DEVICE)
opt = torch.optim.Adam(masker.parameters(), 1e-3)
t0 = time.time()
for epoch in range(2):
    for j in batches(Xtr):
        with torch.no_grad():
            c = clf(Xtr[j]).argmax(1)
        loss = rts_loss(masker(Xtr[j]), Xtr[j], c).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
masker.eval()
print(f'training the masking network, 2 epochs: {time.time() - t0:.0f} s on {DEVICE}')

with torch.no_grad():
    M = masker(Xte)
    kept = (clf(Xte * M).argmax(1) == pred_te).float().mean().item()
    broken = (clf(Xte * (1 - M)).argmax(1) != pred_te).float().mean().item()
print(f'MNIST test set, 10,000 digits: mean mask area {M.mean():.3f}')
print(f'   class preserved under the mask: {kept:.3f}; class changed when the mask is closed: {broken:.3f}')

Now the same objective, but the mask is optimized for every digit separately, as in section 1: the parameter is a $7\times7$ grid before the sigmoid, 300 Adam steps. On the first 20 test digits we compare the time per explanation and the value of $L(M)$ each way reaches.

In [ ]:
def optimize_one(xb, c, steps=300):
    p = torch.zeros(1, 1, 7, 7, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([p], 0.1)
    for _ in range(steps):
        m = F.interpolate(torch.sigmoid(p), size=(28, 28), mode='bilinear', align_corners=False)
        loss = rts_loss(m, xb, c).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return F.interpolate(torch.sigmoid(p), size=(28, 28), mode='bilinear', align_corners=False).detach()

N = 20
torch.manual_seed(0)
t0 = time.time()
M_opt = torch.cat([optimize_one(Xte[i:i + 1], pred_te[i:i + 1]) for i in range(N)])
t_opt = (time.time() - t0) / N
t0 = time.time()
with torch.no_grad():
    M_rts = masker(Xte[:N])
t_rts = (time.time() - t0) / N
with torch.no_grad():
    loss_opt = rts_loss(M_opt, Xte[:N], pred_te[:N]).mean().item()
    loss_rts = rts_loss(M_rts, Xte[:N], pred_te[:N]).mean().item()
print(f'{"":30}{"time per explanation":>22}{"L(M), mean over 20":>22}')
print(f'{"optimization per digit":30}{t_opt * 1e3:>19.1f} ms{loss_opt:>22.3f}')
print(f'{"masking network, one pass":30}{t_rts * 1e3:>19.3f} ms{loss_rts:>22.3f}')

The masking network answers three orders of magnitude faster, but its $L(M)$ is higher than that of optimization per digit: one network for all inputs does not find the best mask for every digit. This gap is called the amortization gap. The second caveat is the tie: the masking network is trained through the gradient of the specific model `clf` on specific data, and for another model it is trained anew.

**Task 3.** For the first eight test digits draw three rows: the digit, the mask of the masking network `M_rts` and the mask of the optimization `M_opt`. For these twenty digits compute the Pearson correlation of the two masks digit by digit and print its median. How many times faster than the optimization is the masking network on your device?

In [ ]:
# Your code here

## 5. L2X: choose $k$ blocks

L2X ([Chen, Song, Wainwright, and Jordan, 2018](https://arxiv.org/abs/1802.07814)) also amortizes, but instead of a smooth mask it chooses exactly $k$ features. The features here are $4\times4$ pixel blocks, and a $28\times28$ digit has 49 of them. Two networks are trained:

- the explainer $w_\theta(X)$ outputs 49 logits; $k$ independent Concrete variables are drawn from them at temperature $\tau=0.1$, as in the paper, and combined by an elementwise maximum, $V_i=\max_j C^j_i$, which gives a soft $k$-hot vector that the gradient passes through;
- the approximator $q$ (the same `SmallCNN` architecture) predicts the answer of the explained model from the input $X\odot V$. Its cross-entropy is the variational lower bound on the mutual information $I(X_S;Y)$ from the lesson “Learnable baselines”. Both networks are trained by RMSprop with step 0.001, as in the paper.

At test time the choice is hard: the $k$ blocks with the largest logits. The quality of the choice is measured by post-hoc accuracy: the share of digits on which a model, from the chosen blocks alone (the rest is the zero background), gives the same answer as from the whole digit. We compute it twice — for the explained model `clf` and for the approximator $q$ — and, for comparison, for $k$ random blocks.

In [ ]:
PATCH, SIDE = 4, 7          # 4x4 blocks, a 7x7 grid, 49 features

class Explainer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(64, 1, 1))

    def forward(self, x):
        return self.net(x).flatten(1)          # (B, 49) — the logits w_theta(X)

def to_pixels(v):
    return v.view(-1, 1, SIDE, SIDE).repeat_interleave(PATCH, 2).repeat_interleave(PATCH, 3)

def concrete_khot(logits, k, tau):
    """k independent Concrete variables and an elementwise maximum: a soft k-hot vector V."""
    u = torch.rand(logits.shape[0], k, logits.shape[1], device=DEVICE).clamp_min(1e-10)
    gumbel = -torch.log(-torch.log(u))
    return ((logits[:, None, :] + gumbel) / tau).softmax(-1).amax(1)

def hard_khot(logits, k):
    return torch.zeros_like(logits).scatter(1, logits.topk(k, 1).indices, 1.0)

def train_l2x(k, tau=0.1, epochs=1, seed=0):
    torch.manual_seed(seed)
    explainer, approx = Explainer().to(DEVICE), SmallCNN().to(DEVICE)
    opt = torch.optim.RMSprop(list(explainer.parameters()) + list(approx.parameters()), 1e-3)   # as in the paper
    for epoch in range(epochs):
        for j in batches(Xtr):
            with torch.no_grad():
                c = clf(Xtr[j]).argmax(1)
            v = concrete_khot(explainer(Xtr[j]), k, tau)
            loss = F.cross_entropy(approx(Xtr[j] * to_pixels(v)), c)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return explainer.eval(), approx.eval()

def evaluate_l2x(explainer, approx, k):
    with torch.no_grad():
        chosen = Xte * to_pixels(hard_khot(explainer(Xte), k))
        random_k = Xte * to_pixels(hard_khot(torch.rand(len(Xte), SIDE * SIDE, device=DEVICE), k))
        return {'clf, chosen blocks': (clf(chosen).argmax(1) == pred_te).float().mean().item(),
                'q, chosen blocks': (approx(chosen).argmax(1) == pred_te).float().mean().item(),
                'clf, random blocks': (clf(random_k).argmax(1) == pred_te).float().mean().item()}

K = 4
t0 = time.time()
explainer4, approx4 = train_l2x(K)
print(f'training L2X, k = {K}, one epoch: {time.time() - t0:.0f} s on {DEVICE}')
for name, acc in evaluate_l2x(explainer4, approx4, K).items():
    print(f'   post-hoc accuracy, {name:20} {acc:.3f}')

In [ ]:
with torch.no_grad():
    sel = to_pixels(hard_khot(explainer4(Xte[:8]), K))
fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for i in range(8):
    axes[0, i].imshow(Xte[i, 0].cpu().numpy(), cmap='gray')
    axes[1, i].imshow((Xte[i, 0] * (0.25 + 0.75 * sel[i, 0])).cpu().numpy(), cmap='gray')
    axes[1, i].set_title(f'clf answer: {pred_te[i].item()}', fontsize=9)
    for row in (0, 1):
        axes[row, i].axis('off')
plt.suptitle(f'L2X: the {K} chosen 4x4 blocks are bright, the rest is dimmed', fontsize=10)
plt.tight_layout()
plt.show()

The chosen blocks preserve the answer of the explained model more often than the same number of random blocks (the first row against the third) — the explainer has learned where the decisive strokes of a digit are. But the approximator $q$ guesses the answer of `clf` from the same blocks more often than `clf` itself does (the second row against the first). The explainer was trained together with $q$ and chose blocks that are readable for $q$ in particular: $q$ saw only such cropped inputs and learned to read them, while `clf` was trained on whole digits. So the chosen subset describes the pair “explainer and $q$”, and the more $q$ differs from the explained model, the less it says about `clf`. This is the limitation from the method card, and the post-hoc accuracy by `clf` is a way to measure it.

**Task 4.** Train L2X at $k=2$ and $k=8$ (the function `train_l2x`) and put the three post-hoc accuracies for $k=2,4,8$ into one table. How does the gap between the accuracy by $q$ and by `clf` change as $k$ grows? Which $k$ would you choose if the explanation has to preserve the answer of `clf` itself on at least 80% of the digits?

In [ ]:
# Your code here

## What to take away from the notebook

- **The mask is a trainable parameter, the method is a short optimization loop.** Meaningful perturbation as a whole is the function `deletion_game`: a mask, a blend with the baseline, an area penalty, Adam.
- **Regularization is part of the definition of the method.** A full-resolution mask without penalties solves the optimization problem with an adversarial pattern; a coarse grid, total variation and jitter turn it into an explanation.
- **$\lambda$ or the area.** Meaningful perturbation gives a family of masks over $\lambda$, Extremal perturbations over an area set in understandable units. The deletion game and the preservation game do not have to give the same region.
- **A metric depends on its perturbation.** By deletion with blur occlusion leads, and by insertion Extremal perturbations leads and occlusion is behind both optimization maps: the masks of Meaningful perturbation and Extremal perturbations are optimal for their own perturbation and their own areas, not for the pixel order the metric walks through.
- **Amortization trades time per explanation for training.** The Real-time saliency masking network answers in one pass but loses to per-input optimization in the value of the objective and is tied to its model.
- **L2X describes its approximator.** The post-hoc accuracy by the explained model is lower than by the approximator: the chosen subset has to be checked on the model being explained.

**Labels:** local, post-hoc. Meaningful perturbation, Extremal perturbations and Real-time saliency are model-specific, a gradient with respect to the mask is needed; L2X is model-agnostic, only the answers of the model are needed. Output: a mask the size of the input (or $k$ chosen features for L2X).